In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys

from utils_notebook import format_syngen_adj_indices, get_token_indices, plot_images, visu_mu, visu_variance

# Get the current working directory
cwd = os.getcwd()

# Get the parent directory
parent_dir = os.path.dirname(cwd)
sys.path.append(parent_dir)

import copy
import logging

import torch
from diffusers import DDPMScheduler

from src.models.gsn_config import DistribConfig, GsngConfig, IterefConfig
from src.models.gsn_criterion import IOUGSN, AttendAndExciteGSN, SynGen
from src.models.pipeline import StableDiffusion3PipelineGSN, StableDiffusionGSN

logging.basicConfig(stream=sys.stdout, level=logging.INFO)
logger = logging.getLogger(__name__)
logger.setLevel("INFO")


if torch.cuda.is_available():
    device = torch.device("cuda")
elif (
    getattr(torch.backends, "mps", None) is not None
    and torch.backends.mps.is_available()
    and torch.backends.mps.is_built()
):
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"on {device}")

## SD 14

In [ ]:
dtype = torch.float32
model_id = "CompVis/stable-diffusion-v1-4"
# model_id = "segmind/tiny-sd" # Small model for testing the pipeline, not used for the experiments in the paper
pipeline = StableDiffusionGSN.from_pretrained(model_id, torch_dtype=dtype)
pipeline.safety_checker = None
pipeline.to(device)
config_scheduler = copy.copy(pipeline.scheduler.config)
print(f"on {device}")
pipeline.scheduler = DDPMScheduler.from_config(config_scheduler)

In [ ]:
prompt = "a photo of a dog and a cat"
entity_list = ["dog", "cat"]
adj_list = None
start_idx_clip = 4  # For the Syngen criterion, the first 4 tokens are "a photo of a" which are not relevant for the criterion. This is specific to this prompt and should be adapted for other prompts.

## Configure the generation

In [ ]:
token_indices, adj_indices, last_idx = get_token_indices(pipeline, prompt, entity_list, adj_list)

criterion_gsng = SynGen()
extra_params_gsng = {
    "adjs_token_clip": format_syngen_adj_indices(criterion_gsng, adj_indices),
    "token_indices_clip": [token_indices],
    "start_idx_clip": [start_idx_clip],
}

criterion_distrib = IOUGSN()
extra_params_distrib = {"token_indices_clip": [token_indices]}

intervention_step_distrib = 10
distrib_config = DistribConfig(
    batch_size_noise=10,
    max_opti=50,
    step_size=20,
    init_mu="x0_mean",
    rescale=True,
    momentum_saga=0.1,
    block=0,
    log_var=True,
    per_channel=True,
    criterion=criterion_distrib,
    step=intervention_step_distrib,
    optimizer_class="SGD",
    extra_params=extra_params_distrib,
    one_image_per_distrib=False,
)

gsng_config = GsngConfig(
    scale_factor=20,
    scale_range=[1, 1],
    steps=list(range(intervention_step_distrib, 25)),
    criterion=criterion_gsng,
    extra_params=extra_params_gsng,
)


# example of adding attend and excite iteref
criterion_iteref = AttendAndExciteGSN()
iteref_config = IterefConfig(
    criterion=criterion_iteref,
    max_opti=50,
    scale_factor=20,
    scale_range=[1, 0.5],
    thresholds={0: 0.05, 10: 0.5, 20: 0.8},
    extra_params={"token_indices_clip": [token_indices]},
    steps=[0, 10, 20],
)


# distrib_config = None
# gsng_config = None
iteref_config = None

In [ ]:
return_intermediate_features = True  # get all the intermediate values as pt
decode_all = True  # to return intermediate latent as image
path_attention = None
print(f"Generation will be done with GSN guidance : {gsng_config}, ITEREF {iteref_config}, SAGA {distrib_config}")

In [ ]:
# Seeds for the latent noise used to draw the latent images inside the learned distribution
# Or in general, seeds for the generation of the images (if using distrib or not)
seeds = [0, 1, 2, 3]
seeds = [0]
generator = [torch.Generator(device=device).manual_seed(seed) for seed in seeds]

# Seeds for the learning of the distribution (if using distrib)
seeds_distrib = [0]
distrib_seed_generators = [torch.Generator(device=device).manual_seed(s) for s in seeds_distrib]


output = pipeline(
    prompt=prompt,
    generator=generator,
    num_images_per_prompt=len(generator),
    num_inference_steps=50,
    return_intermediate_features=return_intermediate_features,
    decode_all=decode_all,
    store_attention_path=path_attention,
    # iteref
    iteref_config=iteref_config,
    # GSNg
    gsng_config=gsng_config,
    # distrib
    distrib_config=distrib_config,
    generator_distrib=distrib_seed_generators,
)

if len(output) == 2:
    output, intermediate_values = output

In [ ]:
plot_images(output.images, size_img=256)

In [ ]:
# Visualize the intermediate values for the first image in the batch
visu_mu(intermediate_values, 0)

In [ ]:
# Visualize the learned variance for the first image in the batch
visu_variance(intermediate_values, 0)

## SD 3

In [ ]:
dtype = torch.bfloat16
pipeline = StableDiffusion3PipelineGSN.from_pretrained(
    "stabilityai/stable-diffusion-3-medium-diffusers",
    torch_dtype=dtype,
)
pipeline.safety_checker = None
pipeline.to(device)
config_scheduler = copy.copy(pipeline.scheduler.config)
print(f"on {device}")
pipeline.enable_model_cpu_offload()

In [ ]:
prompt = "a photo of a cat next to a dog and a giraffe and a dolphin"
entity_list = ["cat", "dog", "giraffe", "dolphin"]
adj_list = None
start_indices = 4

## Configure the generation

In [ ]:
(token_indices_clip, token_indices_t5), (adj_indices_clip, adj_indices_t5), (last_idx_clip, last_idx_t5) = (
    get_token_indices(pipeline, prompt, entity_list, adj_list)
)

criterion_distrib = IOUGSN()
extra_params_distrib = {
    "token_indices_clip": [token_indices_clip],
    "token_indices_t5": [token_indices_t5],
    "last_idx_clip": last_idx_clip,
    "last_idx_t5": last_idx_t5,
}

intervention_step_distrib = 5

distrib_config = DistribConfig(
    # batch_size_noise=10,
    batch_size_noise=4,
    max_opti=50,
    step_size=20,
    init_mu="x0_mean",
    rescale=True,
    momentum_saga=0.7,
    block=0,
    log_var=True,
    per_channel=True,
    criterion=criterion_distrib,
    step=intervention_step_distrib,
    optimizer_class="SGD",
    extra_params=extra_params_distrib,
    one_image_per_distrib=False,
)

# distrib_config = None
gsng_config = None
iteref_config = None

In [ ]:
return_intermediate_features = False  # get all the intermediate values as pt
decode_all = True  # to return intermediate latent as image
one_image_per_distrib = False

path_attention = None
print(f"Generation will be done with GSN guidance : {gsng_config}, ITEREF {iteref_config}, SAGA {distrib_config}")

In [ ]:
# Seeds for the latent noise used to draw the latent images inside the learned distribution
# Or in general, seeds for the generation of the images (if using distrib or not)
seeds = [0, 1, 2, 3]
generator = [torch.Generator(device=device).manual_seed(seed) for seed in seeds]

# Seeds for the learning of the distribution (if using distrib)
seeds_distrib = [0]
distrib_seed_generators = [torch.Generator(device=device).manual_seed(s) for s in seeds_distrib]

output = pipeline(
    prompt=prompt,
    generator=generator,
    num_images_per_prompt=len(generator),
    return_intermediate_features=return_intermediate_features,
    decode_all=decode_all,
    store_attention_path=path_attention,
    # iteref
    iteref_config=iteref_config,
    # GSNg
    gsng_config=gsng_config,
    # distrib
    distrib_config=distrib_config,
    generator_distrib=distrib_seed_generators,
    num_inference_steps=28,
    height=768,
    width=768,
)

if len(output) == 2:
    output, intermediate_values = output

In [ ]:
plot_images(output.images, size_img=256)